## Website Scrape & Ingestion

Crawl `rc.virginia.edu` + `learning.rc.virginia.edu` + `archive.rc.virginia.edu` and convert into markdown files for knowledge base

Pipeline consistent with jira-cloud-full / md-new / video-v2:
- Metadata: only `source_type`, `source`, `chunk_number` (no `tags`/`date_updated`)

## 2. Crawl Websites

Recursively crawl all three domains. Extracts text from `<article>` elements, strips images and metadata tags.

In [ ]:
import cloudscraper
from bs4 import BeautifulSoup
import re
from urllib.parse import urlparse, urljoin
import time


SKIP_EXTENSIONS = (".png", ".jpg", ".jpeg", ".gif", ".bmp", ".svg", ".pdf", ".zip", ".tar", ".gz", ".mp4", ".webp")
ALLOWED_NETLOCS = {"rc.virginia.edu", "learning.rc.virginia.edu", "archive.rc.virginia.edu"}

scraper = cloudscraper.create_scraper()


def is_valid(url, allowed_netlocs=ALLOWED_NETLOCS):
    parsed = urlparse(url)
    path = parsed.path.lower()
    return (
        parsed.scheme in {"http", "https"}
        and parsed.netloc in allowed_netlocs
        and not path.endswith(SKIP_EXTENSIONS)
    )

def crawl(url, visited=set(), documents={}, netloc=None):
    if url in visited:
        return documents
    visited.add(url)

    netloc = netloc or urlparse(url).netloc

    try:
        response = scraper.get(url, timeout=15)

        if response.url != url:
            url = response.url
            if url in visited:
                return documents
            visited.add(url)

        content_type = response.headers.get("Content-Type", "")
        if "text/html" not in content_type:
            return documents
        if response.status_code != 200:
            return documents

        soup = BeautifulSoup(response.text, "html.parser")

        if len(articles := soup.find_all("article")) != 1:
            print(f"Skipping {url} (no single article)")
        else:
            article_soup = BeautifulSoup(str(articles[0]), "html.parser")

            for tag in article_soup.find_all("img"):
                tag.decompose()

            return_link = article_soup.find("a", string=re.compile(r"^\u00ab Return to"))
            if return_link:
                return_link.decompose()

            metadata_tag = article_soup.find("p", class_="blog-post-meta")

            for a_tag in article_soup.find_all("a", href=True):
                href = a_tag["href"]
                if not href.startswith("http"):
                    href = urljoin(url, href)
                a_tag["href"] = href
                a_tag.string = f"[{a_tag.get_text(strip=True)}]({href})"

            title_tag = article_soup.find("h2", class_="blog-post-title")
            if title_tag:
                title_text = title_tag.get_text(strip=True)
                title_tag.string = f"# {title_text}\n\n"

            for h1_tag in article_soup.find_all("h1"):
                h1_text = h1_tag.get_text(strip=True)
                h1_tag.string = f"\n\n## {h1_text}\n"

            if metadata_tag:
                metadata_tag.decompose()

            text = article_soup.get_text()
            text = re.sub(r"https?:\/\/\S+?\.png", "", text)
            text = re.sub(r"\S+\.png", "", text)
            text = re.sub(r"\n{3,}", "\n\n", text)
            text = re.sub(r"[ \t]+", " ", text)

            documents[url] = {
                "text": text.strip(),
                "source": url,
            }
            print(f"Extracted: {url}")

        for a_tag in soup.find_all("a", href=True):
            next_url = a_tag["href"]
            if not next_url.startswith(("http://", "https://")):
                next_url = urljoin(url, next_url)
            next_url = next_url.split("#")[0]

            if next_url not in visited and is_valid(next_url):
                crawl(next_url, visited, documents, netloc)

        time.sleep(0.2)

    except Exception as e:
        print(f"Failed to crawl {url}: {e}")

    return documents

In [ ]:
documents = crawl("https://rc.virginia.edu/")
#documents.update(crawl("https://learning.rc.virginia.edu/"))
print(f"\nTotal pages crawled: {len(documents)}")

## 3. Sitemap Gap Fill

Fetch URLs from sitemaps that the link-following crawler missed.

In [ ]:
import xml.etree.ElementTree as ET

SKIP_PATTERNS = ['/author/', '/category/', '/tag/']

def get_sitemap_urls(sitemap_url, skip_patterns=None):
    skip_patterns = skip_patterns or []
    ns = {"sm": "http://www.sitemaps.org/schemas/sitemap/0.9"}

    print(f"Fetching sitemap: {sitemap_url}")

    resp = scraper.get(sitemap_url, timeout=15)
    resp.raise_for_status()

    print(f"  Status: {resp.status_code}")
    print(f"  Content-Type: {resp.headers.get('Content-Type')}")

    try:
        root = ET.fromstring(resp.text)
    except ET.ParseError as e:
        print(f"  Could not parse sitemap XML: {e}")
        print(f"  Response starts with: {resp.text[:300]!r}")
        return []

    base = "https://" + sitemap_url.split("/")[2]

    raw = []

    for url in root.findall("sm:url", ns):
        loc = url.find("sm:loc", ns)

        if loc is not None and loc.text:
            raw.append(loc.text.strip())

    urls = [
        base + u if u.startswith("/") else u
        for u in raw
    ]

    urls = [
        u for u in urls
        if not any(pattern in u for pattern in skip_patterns)
    ]

    print(f"  Found {len(urls)} URLs")

    return urls


def extract_article(url):
    response = scraper.get(url, timeout=15)
    if response.status_code != 200 or 'text/html' not in response.headers.get('Content-Type', ''):
        return None
    soup = BeautifulSoup(response.text, 'html.parser')
    articles = soup.find_all('article')
    if len(articles) != 1:
        return None
    article_soup = BeautifulSoup(str(articles[0]), 'html.parser')
    for tag in article_soup.find_all('img'):
        tag.decompose()
    metadata_tag = article_soup.find('p', class_='blog-post-meta')
    for a_tag in article_soup.find_all('a', href=True):
        href = a_tag['href']
        if not href.startswith('http'):
            href = urljoin(url, href)
        a_tag['href'] = href
        a_tag.string = '[' + a_tag.get_text(strip=True) + '](' + href + ')'
    if metadata_tag:
        metadata_tag.decompose()
    text = article_soup.get_text()
    text = re.sub(r'https?:\/\/\S+?\.png', '', text)
    text = re.sub(r'\S+\.png', '', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return {'text': text.strip(), 'source': url}

rc_urls    = get_sitemap_urls('https://rc.virginia.edu/sitemap.xml')
learn_urls = get_sitemap_urls('https://learning.rc.virginia.edu/sitemap.xml', SKIP_PATTERNS)
all_sitemap_urls = rc_urls + learn_urls

missing = [u for u in all_sitemap_urls if u not in documents]
print(f"Sitemap total: {len(all_sitemap_urls)}, already crawled: {len(all_sitemap_urls)-len(missing)}, missing: {len(missing)}")

for url in missing:
    try:
        result = extract_article(url)
        if result:
            documents[url] = result
            print(f"Added: {url}")
        else:
            print(f"Skipped (no article): {url}")
        time.sleep(0.2)
    except Exception as e:
        print(f"Failed {url}: {e}")

print(f"\nTotal documents after gap fill: {len(documents)}")

## 3b. Patch JS-rendered Pages

Some pages (e.g. the Slurm Script Generator) are interactive JavaScript tools that the static scraper cannot render, resulting in empty or title-only content. Manually supply descriptive text for these pages so they can be retrieved by the RAG pipeline.

In [ ]:
manual_patches = {
    "https://rc.virginia.edu/userinfo/hpc/slurm-script-generator/": {
        "text": """# Slurm Script Generator

The UVA Research Computing Slurm Script Generator is an interactive web tool that helps users create Slurm job submission scripts for the UVA HPC system.

## How to Use

Visit the Slurm Script Generator at: https://archive.rc.virginia.edu/userinfo/hpc/slurm-script-generator/

The tool allows you to:
- Select a partition (queue) for your job (e.g., standard, gpu, parallel, dev)
- Specify the number of nodes and cores (tasks) needed
- Set memory requirements per core or per node
- Configure wall time (how long the job will run)
- Set up GPU resources if needed (number and type of GPUs)
- Specify your allocation group
- Add email notifications for job start, end, or failure
- Generate a ready-to-use Slurm batch script (.slurm file)

## When to Use This Tool

Use the Slurm Script Generator if you are:
- New to HPC and need help writing your first Slurm script
- Unsure about which Slurm directives (#SBATCH) to include
- Looking for a quick way to generate a template job script
- Wanting to explore what options are available for different partitions

## Example Output

The generator produces a script like:
```
#!/bin/bash
#SBATCH --job-name=myjob
#SBATCH --partition=standard
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=4
#SBATCH --mem=16G
#SBATCH --time=02:00:00
#SBATCH -A mygroup

module load your_software
your_command_here
```

## Related Resources

- Slurm Job Manager documentation: https://rc.virginia.edu/userinfo/hpc/slurm/
- HPC Getting Started: https://rc.virginia.edu/getting-started
- Sample Slurm scripts by partition: https://rc.virginia.edu/userinfo/hpc/slurm/
""",
        "source": "https://rc.virginia.edu/userinfo/hpc/slurm-script-generator/",
    },
    "https://rc.virginia.edu/userinfo/hpc/software/physics/": {
        "text": """# Physics Software on UVA HPC

UVA Research Computing provides several physics simulation and computational physics software packages on the HPC system.

## Available Physics Software

The following physics-related software is available via the module system on UVA HPC:

- **LAMMPS** - Large-scale Atomic/Molecular Massively Parallel Simulator for molecular dynamics
- **GROMACS** - Molecular dynamics package for simulating proteins, lipids, and nucleic acids
- **Quantum ESPRESSO** - Integrated suite for electronic-structure calculations and materials modeling
- **VASP** - Vienna Ab initio Simulation Package for atomic scale materials modelling
- **OpenFOAM** - Computational fluid dynamics (CFD) toolbox
- **NAMD** - Parallel molecular dynamics for large biomolecular systems

## How to Use

Load physics software using the module system:
```
module load lammps
module load gromacs
module load quantumespresso
```

Use `module spider <software_name>` to see available versions and dependencies.

## Related Resources

- Full software list: https://rc.virginia.edu/userinfo/hpc/software/
- Slurm job submission: https://rc.virginia.edu/userinfo/hpc/slurm/
""",
        "source": "https://rc.virginia.edu/userinfo/hpc/software/physics/",
    },
}

patched = 0
for url, doc in manual_patches.items():
    if url not in documents or len(documents.get(url, {}).get("text", "")) < 100:
        documents[url] = doc
        patched += 1
        print(f"Patched: {url}")
    else:
        print(f"Already has content: {url}")

print(f"\nManually patched {patched} JS-rendered pages.")
print(f"Total documents: {len(documents)}")

## 5. Preview

In [ ]:
for url, doc in list(documents.items())[:3]:
    print(f"=== {url} ===")
    print(doc["text"][:300])
    print()

## 3. Generate Markdown Files

In [ ]:
from pathlib import Path
import re

# Locate this repo from either the project root or a notebook subfolder.
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "app" / "kb_integration" / "tasks.py").is_file()
     and (path / "scrapers").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or a subfolder.")

# Store generated files inside this repo
OUTPUT_FOLDER = PROJECT_ROOT / "data" / "website"

# Automatically create data/website if it doesn't exist
OUTPUT_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)


def safe_filename(url):

    filename = re.sub(
        r'[^a-zA-Z0-9_\-]', 
        '_', 
        url
    )

    filename = re.sub(
        r'_+', 
        '_', 
        filename
    ).strip('_')

    return filename[:150]


print(
    f"Documents available: "
    f"{len(documents)}"
)

print(
    f"Writing files to: "
    f"{OUTPUT_FOLDER}"
)


created = 0
failed = 0


for url, doc in documents.items():

    try:

        # Keep the Markdown extension so the uploader discovers this file.
        flat_name = f"{safe_filename(url)}.md"
        file_path = OUTPUT_FOLDER / flat_name

        with open(
            file_path,
            "w",
            encoding="utf-8"
        ) as f:

            f.write(
                doc["text"].strip() + "\n"
            )

        created += 1

        print(
            f"Created: {file_path.name}"
        )

    except Exception as e:

        failed += 1

        print(
            f"ERROR processing "
            f"{url}: "
            f"{e}"
        )


print("=" * 60)
print("MARKDOWN GENERATION COMPLETE")
print("=" * 60)

print(f"Created: {created} files")
print(f"Failed:  {failed} files")
print(f"Output folder: {OUTPUT_FOLDER}")